In [4]:
from collections.abc import Mapping, Sequence

import numpy as np
import polars as pl


def _get_bins(
    df: pl.DataFrame, n_bins: Mapping[str, int | Sequence[float]]
) -> dict[str, tuple[float, float]]:
    """Learns the bin edges of each column from the training data.

    Args:
        df (pl.DataFrame): Training data.
        n_bins (Mapping[str, int | Sequence[float]]): Per column, the number of
            equally filled bins, or the quantiles to split at (like `qcut`).

    Returns:
        dict[str, list[float]]: Each column's inner bin edges, sorted, with
            duplicates merged (e.g. when a quarter of the values are 0).
    """
    quantiles = {
        col: np.linspace(0, 1, spec + 1)[1:-1].tolist()
        if isinstance(spec, int)
        else list(spec)
        for col, spec in n_bins.items()
    }
    return df.select(
        pl.concat_list(pl.col(col).quantile(q, interpolation="linear") for q in qs)
        .list.unique()
        .list.sort()
        .alias(col)
        for col, qs in quantiles.items()
    ).row(0, named=True)


df = pl.read_csv("/Users/leon/Desktop/programming/ml-training/data/housing.csv")
bins = {"crim": 5, "age": 10}
_get_bins(df, bins)

{'crim': [0.06417, 0.15038, 0.5500700000000004, 5.58107],
 'age': [26.950000000000003,
  37.8,
  52.400000000000006,
  65.4,
  77.5,
  85.90000000000002,
  91.8,
  95.6,
  98.8]}